In [ ]:
import os
import json
import re
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from collections import defaultdict


def load_json_files(file_path):
    """加载文件夹中所有符合命名规则的JSON文件"""

    temp=[]
    try:
        def stream_json_lines(filename):
            with open(filename, 'r', encoding='utf-8') as f:
                for line in f:
                    if line.strip():
                        yield json.loads(line)
        # 使用示例
        for item in stream_json_lines(file_path):
            temp.append(item)
    
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
    return temp

from statistics import mean
import math
def extract_metrics_allcategory(data):
    """从加载的数据中提取指标"""
    # 只需要两层嵌套：layer → head → list
    metrics = defaultdict(lambda: defaultdict(lambda: defaultdict(list)))

    for i in range(len(data)): # 遍历每个字典，类别_图片id+
        for key, layer_heads in data[i].items(): # 初步拆分，key=person_532481 ,layer_heads={'0_0': [100.0, 0.00918272630834512, 100.0], '0_1': [100.0, 0.009210715908713049, 100.0]等
            for layer_head, params in layer_heads.items():
                layer, head = layer_head.split('_')
                eval1 = params[0]
                eval2 = params[1]
                eval3 = params[2]
                eval4 = params[3]
                metrics[int(layer)][int(head)]['eval1'].extend([eval1])
                metrics[int(layer)][int(head)]['eval2'].extend([eval2])
                metrics[int(layer)][int(head)]['eval3'].extend([eval3])
                metrics[int(layer)][int(head)]['eval4'].extend([eval4])
    auto_param={}
    # 计算所有图片对应层和注意力头的指标平均值
    metrics_mean = defaultdict(lambda: defaultdict(list))
    for layer, heads in metrics.items():
        for head, values in heads.items():
            metrics_mean[layer][head] = [ mean(values['eval1']), mean(values['eval2']), mean(values['eval3']), mean(values['eval4'])]
            auto_param[f"{layer}_{head}"]=mean(values['eval1'])
    return metrics_mean,auto_param


In [ ]:

VI_PATH="/home/user/Guotianxing/code/Fusion_attention/8_27_auto_param_result/Qwen2.5-VL-3B-Instruct_qwen2.5vl_MSRS_Auto_rel_vi.json"
VI_DATA=load_json_files(VI_PATH)
IR_PATH="/home/user/Guotianxing/code/Fusion_attention/8_27_auto_param_result/Qwen2.5-VL-3B-Instruct_qwen2.5vl_MSRS_Auto_rel_ir.json"
IR_DATA=load_json_files(IR_PATH)
vi_file = os.path.basename(VI_PATH).split('.')[0]
ir_file = os.path.basename(IR_PATH).split('.')[0]
model_type=vi_file.split('_')[0] 
metrics_vi,auto_param_vi = extract_metrics_allcategory(VI_DATA)
metrics_ir,auto_param_ir = extract_metrics_allcategory(IR_DATA)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from matplotlib import cm
from matplotlib.lines import Line2D
from matplotlib.ticker import ScalarFormatter, FormatStrFormatter
# 设置中文字体支持
plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams['axes.unicode_minus'] = False  # 解决负号显示问题

eval_type = "Cross Entropy"

def visualize_dual_metrics_3d(metrics_ir, metrics_vi, labels=None):
    """将两个指标数据可视化为三维图及各平面视图"""
    
    if labels is None:
        labels = ['IR Metrics', 'VI Metrics']
    
    # 合并所有的 layer 和 head，确保坐标轴一致
    all_layers = set()
    all_heads = set()
    
    for metrics in [metrics_ir, metrics_vi]:
        all_layers.update(metrics.keys())
        for layer_data in metrics.values():
            all_heads.update(layer_data.keys())
    
    layers = sorted(all_layers)
    heads = sorted(all_heads)
    
    # 创建坐标轴索引映射
    layer_to_idx = {layer: i for i, layer in enumerate(layers)}
    head_to_idx = {head: j for j, head in enumerate(heads)}
    
    # 准备两组数据
    def extract_data(metrics):
        x, y, z = [], [], []
        for layer, layer_data in metrics.items():
            for head, metric_values in layer_data.items():
                # 假设metric_values是(acc, recall, f1)的元组
                acc, recall, f1, ce = metric_values
                x.append(layer_to_idx[layer])
                y.append(head_to_idx[head])
                if eval_type == "Accuracy":
                    z.append(acc)
                elif eval_type == "Recall":
                    z.append(recall)
                elif eval_type == "F1-score":
                    z.append(f1)
                elif eval_type == "Cross Entropy":
                    z.append(ce)
                else:
                    print("ERROR: Invalid eval_type")
                    exit()
        return x, y, z
    
    x1, y1, z1 = extract_data(metrics_ir)
    x2, y2, z2 = extract_data(metrics_vi)
    
    # 创建图形
    fig = plt.figure(figsize=(20, 15))
    
    # 3D散点图
    ax_3d = fig.add_subplot(221, projection='3d')
    
    # 绘制第一组数据 (IR) - 红色系
    scatter1_3d = ax_3d.scatter(x1, y1, z1, c=z1, cmap=cm.Reds, s=100, 
                               alpha=0.8, label=labels[0], marker='o')
    
    # 绘制第二组数据 (VI) - 蓝色系  
    scatter2_3d = ax_3d.scatter(x2, y2, z2, c=z2, cmap=cm.Blues, s=100, 
                               alpha=0.8, label=labels[1], marker='s')
    
    # 添加颜色条

    # 设置第一个颜色条及其文字大小
    cbar1_3d = fig.colorbar(scatter1_3d, ax=ax_3d, pad=0.1, shrink=0.35, 
                            label=f'{labels[0]} {eval_type}')
    # 设置颜色条标签文字大小
    cbar1_3d.set_label(label=f'{labels[0]} {eval_type}', fontsize=18)  # 标签文字大小
    # 设置颜色条刻度文字大小
    cbar1_3d.ax.tick_params(labelsize=13)  # 刻度文字大小
    cbar1_3d.ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))  # 2位小数
    # 设置第二个颜色条及其文字大小
    cbar2_3d = fig.colorbar(scatter2_3d, ax=ax_3d, pad=0.15, shrink=0.35, 
                            label=f'{labels[1]} {eval_type}')
    # 设置颜色条标签文字大小
    cbar2_3d.set_label(label=f'{labels[1]} {eval_type}', fontsize=18)  # 标签文字大小
    # 设置颜色条刻度文字大小
    cbar2_3d.ax.tick_params(labelsize=13)  # 刻度文字大小
    cbar2_3d.ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))  # 2位小数

    # 设置x轴、y轴、z轴的标签文字大小
    ax_3d.set_xlabel(f'Layer (0-{len(layers)})', rotation=5, fontsize=20)
    ax_3d.set_ylabel(f'Head (0-{len(heads)})', rotation=-30, fontsize=20)
    ax_3d.set_zlabel(f'{eval_type}', fontsize=20)
    
    # 获取总刻度值的范围
    ax_3d.set_xticks(np.arange(0, len(layers), 1))  # 生成从0到32的刻度，步长为8
    ax_3d.set_yticks(np.arange(0, len(heads), 1))
    # 设置是否显示标签
    # ax_3d.set_xticklabels(layers, rotation=45, ha='right')
    # ax_3d.set_yticklabels(heads)
    ax_3d.set_xticklabels([])  # 不显示x轴的刻度标签
    ax_3d.set_yticklabels([])  # 不显示y轴的刻度标签
    # 设置z轴刻度文字大小
    ax_3d.tick_params(axis='z', labelsize=13)  # 调整labelsize数值控制大小
    # ax_3d.set_title(f'3D Comparison of {eval_type}', fontsize=20)
    legend_elements = [Line2D([0], [0], marker='s', color='w', markerfacecolor='navy', 
                        markersize=11, alpha=0.7, label=labels[1]),
                    Line2D([0], [0], marker='o', color='w', markerfacecolor='red', 
                    markersize=11, alpha=0.5, label=labels[0])]
    ax_3d.legend(handles=legend_elements, fontsize=13)
    ax_3d.view_init(elev=30, azim=45)
    
    # XY平面视图 - 叠加显示（根据z值着色）
    ax_xy = fig.add_subplot(222)

    # 使用z值作为颜色映射的依据
    scatter1_xy = ax_xy.scatter(x1, y1, c=z1, s=100, alpha=1, 
                            label=labels[0], marker='o', 
                            cmap='Reds', vmin=min(np.min(z1), np.min(z2)), 
                            vmax=max(np.max(z1), np.max(z2)))

    scatter2_xy = ax_xy.scatter(x2, y2, c=z2, s=100, alpha=0.2, 
                            label=labels[1], marker='s',  # 使用不同形状区分
                            cmap='Blues', vmin=min(np.min(z1), np.min(z2)), 
                            vmax=max(np.max(z1), np.max(z2)))

    ax_xy.set_xlabel('Layer', fontsize=22)
    ax_xy.set_ylabel('Head', fontsize=22)
    # ax_xy.set_title(f'XY Plane View Comparison (Color by {eval_type})', fontsize=22)
    ax_xy.set_xticks(range(len(layers)))
    ax_xy.set_xticklabels(layers)
    ax_xy.set_yticks(range(len(heads)))
    ax_xy.set_yticklabels(heads)
    ax_xy.grid(True, linestyle='--', alpha=0.7)

    # # 添加颜色条
    # cbar_xy = plt.colorbar(scatter1_xy, ax=ax_xy, shrink=0.8)
    # cbar_xy.set_label(f'{eval_type} Value', fontsize=10)

    # 自定义图例（因为颜色映射会影响原有图例）
    
    legend_elements = [Line2D([0], [0], marker='s', color='w', markerfacecolor='navy', 
                        markersize=11, alpha=0.7, label=labels[1]),
                    Line2D([0], [0], marker='o', color='w', markerfacecolor='red', 
                    markersize=11, alpha=0.5, label=labels[0])]
    ax_xy.legend(handles=legend_elements, fontsize=15)

    # XZ平面视图 - 层vs指标值（根据z值着色）
    ax_xz = fig.add_subplot(223)

    scatter1_xz = ax_xz.scatter(x1, z1, c=z1, marker='o', s=80, alpha=0.8, 
                            label=labels[0], cmap='Reds')
    scatter2_xz = ax_xz.scatter(x2, z2, c=z2, marker='s', s=80, alpha=0.8, 
                            label=labels[1], cmap='Blues')

    ax_xz.set_xlabel(f'Layer (0-{len(layers)})', fontsize=42)
    ax_xz.set_ylabel(f'{eval_type}', fontsize=42)
    # ax_xz.set_title(f'XZ Plane View: Layer - {eval_type}', fontsize=42)
    ax_xz.set_xticklabels([]) 
    ax_xz.set_yticklabels([]) 
    ax_xz.grid(True, linestyle='--', alpha=0.7)
    ax_xz.tick_params(axis='both', labelsize=13)  # 设置x轴和y轴刻度文字大小为18

    # # 添加颜色条
    # cbar_xz = plt.colorbar(scatter1_xz, ax=ax_xz, shrink=0.8)
    # cbar_xz.set_label(f'{eval_type} Value', fontsize=10)

    # 自定义图例
    legend_elements = [Line2D([0], [0], marker='s', color='w', markerfacecolor='navy', 
                        markersize=11, alpha=0.7, label=labels[1]),
                    Line2D([0], [0], marker='o', color='w', markerfacecolor='red', 
                    markersize=11, alpha=0.5, label=labels[0])]
    ax_xz.legend(handles=legend_elements, fontsize=26)

    # YZ平面视图 - 头vs指标值（根据z值着色）
    ax_yz = fig.add_subplot(224)

    scatter1_yz = ax_yz.scatter(y1, z1, c=z1, marker='o', s=80, alpha=0.8, 
                            label=labels[0], cmap='Reds')
    scatter2_yz = ax_yz.scatter(y2, z2, c=z2, marker='s', s=80, alpha=0.8, 
                            label=labels[1], cmap='Blues')

    ax_yz.set_xlabel(f'Head (0-{len(heads)})', fontsize=42)
    ax_yz.set_ylabel(f'{eval_type}', fontsize=42)
    # ax_yz.set_title(f'YZ Plane View: Head - {eval_type}', fontsize=42)
    ax_yz.set_xticks(range(len(heads)))
    ax_yz.set_xticklabels([]) 
    ax_yz.set_yticklabels([]) 
    ax_yz.grid(True, linestyle='--', alpha=0.7)
    ax_yz.tick_params(axis='both', labelsize=13)  # 设置x轴和y轴刻度文字大小为18

    # # 添加颜色条
    # cbar_yz = plt.colorbar(scatter1_yz, ax=ax_yz, shrink=0.8)
    # cbar_yz.set_label(f'{eval_type} Value', fontsize=10)

    # 自定义图例
    legend_elements = [Line2D([0], [0], marker='s', color='w', markerfacecolor='navy', 
                        markersize=11, alpha=0.7, label=labels[1]),
                    Line2D([0], [0], marker='o', color='w', markerfacecolor='red', 
                    markersize=11, alpha=0.5, label=labels[0])]
                    
    ax_yz.legend(handles=legend_elements, fontsize=26)
    
    # 调整布局
    fig.tight_layout()
    
    return fig, [ax_3d, ax_xy, ax_xz, ax_yz]


In [ ]:
fig, axes = visualize_dual_metrics_3d(metrics_ir, metrics_vi, labels=['IR', 'VI'])
plt.savefig(f"./visual_result/{ir_file}_vi_ir_3d_metrics_plot.pdf", format='pdf', bbox_inches='tight', pad_inches=1)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import cm
from typing import Dict, List, Tuple, Optional, Union

def sort_and_plot_metrics(metrics_mean: Dict[str, Dict[str, List[float]]], 
                          output_path: str = None, 
                          figsize: Tuple[int, int] = (20, 6),
                          top_n: int = 10,
                          ylim: Optional[Union[Tuple[float, float], List[Tuple[float, float]]]] = None) -> None:
    """
    对二维字典中的metrics数据进行排序并绘制条形图
    每个柱子的颜色由下到上逐渐变深，渐变基于坐标轴显示区间
    
    参数:
        metrics_mean: 二维字典，格式为metrics_mean[layer][head] = [param1, param2, param3]
        output_path: 可选，图表保存路径，若为None则显示图表
        figsize: 可选，图表尺寸
        top_n: 可选，每个图表显示的最大数据点数
        ylim: 可选，y轴范围设置
              - None: 自动设置y轴范围
              - (min, max): 所有子图使用相同的y轴范围
              - [(min1, max1), (min2, max2), (min3, max3)]: 为每个子图分别设置y轴范围
    """
    # 提取数据并转换为适合排序的格式
    all_data = []
    for layer, heads in metrics_mean.items():
        for head, params in heads.items():
            all_data.append((f"{layer}_{head}", params[0], params[1], params[2], params[3]))
    
    # 按param1、param2、param3、param4分别排序，并取前top_n个
    sorted_by_param1 = sorted(all_data, key=lambda x: x[1], reverse=True)[:top_n]
    sorted_by_param2 = sorted(all_data, key=lambda x: x[2], reverse=True)[:top_n]
    sorted_by_param3 = sorted(all_data, key=lambda x: x[3], reverse=True)[:top_n]
    sorted_by_param4 = sorted(all_data, key=lambda x: x[4], reverse=True)[:top_n]
    
    # 创建图表
    fig, (ax1, ax2, ax3, ax4) = plt.subplots(1, 4, figsize=figsize)
    
    # 颜色映射
    if output_path and "VI" in output_path:
        cmap = plt.cm.Blues
    else:
        cmap = plt.cm.Reds
    
    
    def add_gradient_bar(ax, x_pos, values, cmap, y_min, y_max, num_segments=20):
        """
        绘制渐变色柱状图，基于坐标轴显示区间进行渐变
        """
        for i, val in enumerate(values):
            # 计算在整个y轴范围内的位置比例
            val_ratio = (val - y_min) / (y_max - y_min) if y_max != y_min else 0
            
            # 计算渐变段数（基于数值在y轴范围内的位置）
            segments_to_draw = max(1, int(val_ratio * num_segments))
            
            # 计算每段的实际高度
            segment_height = val / segments_to_draw
            
            # 生成颜色，基于在整个y轴范围内的位置
            # 从底部(y_min对应的颜色)到当前值对应的颜色
            start_color_ratio = 0.3  # 最浅颜色
            end_color_ratio = 0.3 + 0.7 * val_ratio  # 根据在y轴范围内的位置确定最深颜色
            
            colors = cmap(np.linspace(start_color_ratio, end_color_ratio, segments_to_draw))
            
            # 绘制每一段
            for j in range(segments_to_draw):
                bottom = j * segment_height
                ax.bar(x_pos[i], segment_height, bottom=bottom, width=0.8, 
                      color=colors[j], edgecolor='none')
    
    # 处理ylim参数
    def get_ylim(ax, values, index):
        """获取指定子图的y轴范围"""
        if ylim is None:
            # 使用数据范围，稍微扩展一点
            data_min, data_max = min(values), max(values)
            margin = (data_max - data_min) * 0.05
            return data_min - margin, data_max + margin
        elif isinstance(ylim, tuple) and len(ylim) == 2:
            # 所有子图使用相同范围
            return ylim
        elif isinstance(ylim, list) and len(ylim) >= index + 1:
            # 为每个子图分别设置范围
            if isinstance(ylim[index], tuple) and len(ylim[index]) == 2:
                return ylim[index]
            else:
                # 如果指定的索引不是有效的tuple，回退到自动范围
                data_min, data_max = min(values), max(values)
                margin = (data_max - data_min) * 0.05
                return data_min - margin, data_max + margin
        else:
            # 回退到自动范围
            data_min, data_max = min(values), max(values)
            margin = (data_max - data_min) * 0.05
            return data_min - margin, data_max + margin
    
    # 绘制param1的条形图（Accuracy）
    x_pos1 = np.arange(len(sorted_by_param1))
    values1 = [item[1] for item in sorted_by_param1]
    labels1 = [item[0] for item in sorted_by_param1]
    y_min1, y_max1 = get_ylim(ax1, values1, 0)
    add_gradient_bar(ax1, x_pos1, values1, cmap, y_min1, y_max1)
    ax1.set_xticks(x_pos1)
    ax1.set_xticklabels(labels1, rotation=50, ha='right', fontsize=15)
    ax1.tick_params(axis='y', labelsize=15)
    ax1.set_title('Top Accuracy (Top 10)', fontsize=18)
    ax1.set_xlabel('Layer_Head', fontsize=18)
    ax1.set_ylabel('Accuracy', fontsize=18)
    ax1.set_ylim(y_min1, y_max1)
    # ax1.grid(True, alpha=0.3)
    
    # 绘制param2的条形图（Recall）
    x_pos2 = np.arange(len(sorted_by_param2))
    values2 = [item[2] for item in sorted_by_param2]
    labels2 = [item[0] for item in sorted_by_param2]
    y_min2, y_max2 = get_ylim(ax2, values2, 1)
    add_gradient_bar(ax2, x_pos2, values2, cmap, y_min2, y_max2)
    ax2.set_xticks(x_pos2)
    ax2.set_xticklabels(labels2, rotation=50, ha='right', fontsize=15)
    ax2.tick_params(axis='y', labelsize=15)
    ax2.set_title('Top Recall (Top 10)', fontsize=18)
    ax2.set_xlabel('Layer_Head', fontsize=18)
    ax2.set_ylabel('Recall', fontsize=18)
    ax2.set_ylim(y_min2, y_max2)
    # ax2.grid(True, alpha=0.3)
    
    # 绘制param3的条形图（F1-score）
    x_pos3 = np.arange(len(sorted_by_param3))
    values3 = [item[3] for item in sorted_by_param3]
    labels3 = [item[0] for item in sorted_by_param3]
    y_min3, y_max3 = get_ylim(ax3, values3, 2)
    add_gradient_bar(ax3, x_pos3, values3, cmap, y_min3, y_max3)
    ax3.set_xticks(x_pos3)
    ax3.set_xticklabels(labels3, rotation=50, ha='right', fontsize=15)
    ax3.tick_params(axis='y', labelsize=15)
    ax3.set_title('Top F1-score (Top 10)', fontsize=18)
    ax3.set_xlabel('Layer_Head', fontsize=18)
    ax3.set_ylabel('F1-score', fontsize=18)
    ax3.set_ylim(y_min3, y_max3)
    # ax3.grid(True, alpha=0.3)


    # 绘制param4的条形图（Cross Entropy）
    x_pos4 = np.arange(len(sorted_by_param4))
    values4 = [item[4] for item in sorted_by_param4]
    labels4 = [item[0] for item in sorted_by_param4]
    y_min4, y_max4 = get_ylim(ax4, values4, 3)
    add_gradient_bar(ax4, x_pos4, values4, cmap, y_min4, y_max4)
    ax4.set_xticks(x_pos4)
    ax4.set_xticklabels(labels4, rotation=50, ha='right', fontsize=15)
    ax4.tick_params(axis='y', labelsize=15)
    ax4.set_title('Top Cross Entropy (Top 10)', fontsize=18)
    ax4.set_xlabel('Layer_Head', fontsize=18)
    ax4.set_ylabel('Cross Entropy', fontsize=18)
    ax4.set_ylim(y_min4, y_max4)
    # ax4.grid(True, alpha=0.3)

    plt.tight_layout()
    
    # 保存或显示图表
    if output_path:
        plt.savefig(output_path, format='pdf', dpi=300, bbox_inches='tight')
        print(f"图表已保存至: {output_path}")
    else:
        plt.show()


# 使用示例：
"""
# 示例1: 所有子图使用相同的y轴范围，渐变将基于这个范围
sort_and_plot_metrics(metrics_mean, ylim=(0, 1))

# 示例2: 为每个子图分别设置y轴范围，每个子图的渐变基于各自的范围
sort_and_plot_metrics(metrics_mean, ylim=[(0, 1), (0.5, 1), (0.3, 0.9)])

# 示例3: 使用自动范围，渐变基于数据的实际范围
sort_and_plot_metrics(metrics_mean)
"""


In [ ]:

# sort_and_plot_metrics(metrics_vi,ylim=(0.35, 0.50),output_path=f"./visual_result/{vi_file}_Layer_Head_VI_eval3.pdf")
# sort_and_plot_metrics(metrics_ir,ylim=(0.36, 0.43),output_path=f"./visual_result/{ir_file}_Layer_Head_IR_eval3.pdf")
sort_and_plot_metrics(metrics_vi,output_path=f"./visual_result/{vi_file}_Layer_Head_VI_eval3.pdf")
sort_and_plot_metrics(metrics_ir,output_path=f"./visual_result/{ir_file}_Layer_Head_IR_eval3.pdf")